# 07 — Analytics Queries
## Gold Star Schema → Business Insights

**Purpose:** Production-quality SQL analytics on the Gold star schema.
These queries answer real business questions and serve as reference
implementations for BI tools, dashboards, and executive reporting.

**Input  :** `sales_gold.fact_sales`, `dim_customer`, `dim_product`, `dim_region`
**Output :** Analytical result sets (no writes — read-only layer)
**Layer  :** Analytics
**Audience:** Business stakeholders, BI tools, executive dashboards


## 1. Load config

In [0]:
%run ./00_setup_and_config


## 2. Verification of all Gold tables

In [0]:
from pyspark.sql import functions as F

log("INFO", "07_analytics", "Starting analytics layer")

gold_tables = {
    "fact_sales"   : FACT_TABLE,
    "dim_customer" : DIM_CUSTOMER,
    "dim_product"  : DIM_PRODUCT,
    "dim_region"   : DIM_REGION,
}

print("=== GOLD LAYER READINESS CHECK ===\n")
all_ready = True

for name, table in gold_tables.items():
    try:
        count = spark.table(table).count()
        print(f"  ✅ {table:<40} {count:>6,} rows")
    except Exception as e:
        print(f"  ❌ {table:<40} NOT FOUND")
        all_ready = False

if not all_ready:
    raise Exception("One or more Gold tables missing — run Notebooks 05 and 06 first")

print(f"\n  ✅ All Gold tables ready — analytics layer active")
log("INFO", "07_analytics", "All Gold tables verified")


## 3. Queries

### Q1 — Regional Revenue and Profit Performance
*"Which regions drive the most revenue? Which are most profitable?"*

This is typically the first slide in any executive presentation.
Revenue tells you scale. Profit margin tells you health.
A region with high revenue but low margin is a warning sign.

In [0]:
%sql
SELECT
    f.region,
    COUNT(DISTINCT f.order_id)              AS total_orders,
    COUNT(DISTINCT f.customer_id)           AS unique_customers,
    ROUND(SUM(f.sales), 2)                  AS total_sales,
    ROUND(SUM(f.profit), 2)                 AS total_profit,
    ROUND(SUM(f.profit) / SUM(f.sales) * 100, 2)
                                            AS margin_pct,
    ROUND(SUM(f.sales) / COUNT(DISTINCT f.order_id), 2)
                                            AS avg_order_value,
    RANK() OVER (ORDER BY SUM(f.sales) DESC) AS sales_rank
FROM sales_gold.fact_sales f
JOIN sales_gold.dim_region r ON f.region = r.region
GROUP BY f.region
ORDER BY total_sales DESC

### Q2 — Monthly Sales Trend
*"Is the business growing? What does seasonality look like?"*

Time-series analysis is the backbone of business intelligence.
The order_yearmonth column you derived in Silver makes this trivial.
Without that pre-computation, every BI tool would compute YEAR()
and MONTH() on every row for every dashboard load.

In [0]:
%sql
SELECT
    f.order_yearmonth,
    f.order_year,
    f.order_month,
    COUNT(DISTINCT f.order_id)   AS orders,
    ROUND(SUM(f.sales), 2)       AS monthly_sales,
    ROUND(SUM(f.profit), 2)      AS monthly_profit,
    ROUND(AVG(f.profit_margin) * 100, 2) AS avg_margin_pct,
    -- Month-over-month growth using LAG window function
    ROUND(
        (SUM(f.sales) - LAG(SUM(f.sales))
            OVER (ORDER BY f.order_year, f.order_month))
        / LAG(SUM(f.sales))
            OVER (ORDER BY f.order_year, f.order_month) * 100,
        2
    )                            AS mom_growth_pct
FROM sales_gold.fact_sales f
GROUP BY
    f.order_yearmonth,
    f.order_year,
    f.order_month
ORDER BY
    f.order_year,
    f.order_month

### Q3 — Category and Sub-Category Profitability
*"Which product lines should we invest in? Which should we review?"*

Revenue without profit context is misleading.
Technology might have lower sales volume than Furniture
but significantly better margins. This query surfaces that.

In [0]:
%sql
SELECT
    p.category,
    p.sub_category,
    COUNT(*)                                    AS transactions,
    ROUND(SUM(f.sales), 2)                      AS total_sales,
    ROUND(SUM(f.profit), 2)                     AS total_profit,
    ROUND(SUM(f.profit) / SUM(f.sales) * 100, 2) AS margin_pct,
    ROUND(AVG(f.discount) * 100, 1)             AS avg_discount_pct,
    SUM(f.quantity)                             AS units_sold,
    -- Profitability flag for quick filtering
    CASE
        WHEN SUM(f.profit) / SUM(f.sales) >= 0.15 THEN '🟢 Healthy'
        WHEN SUM(f.profit) / SUM(f.sales) >= 0.05 THEN '🟡 Watch'
        WHEN SUM(f.profit) / SUM(f.sales) >= 0    THEN '🟠 Low'
        ELSE                                           '🔴 Loss-making'
    END                                         AS profitability_status
FROM sales_gold.fact_sales   f
JOIN sales_gold.dim_product  p ON f.product_id = p.product_id
GROUP BY
    p.category,
    p.sub_category
ORDER BY
    p.category,
    total_profit DESC

### Q4 — Discount Impact on Profitability
*"Are our discounts destroying our margins?"*

This is one of the most actionable queries in the dataset.
Superstore's data famously shows that high discounts
correlate strongly with negative profit — a business insight
that often surprises stakeholders who assume discounts drive volume.

In [0]:
%sql
SELECT
    f.discount_band,
    COUNT(*)                                     AS transactions,
    ROUND(AVG(f.discount) * 100, 1)              AS avg_discount_pct,
    ROUND(SUM(f.sales), 2)                       AS total_sales,
    ROUND(SUM(f.profit), 2)                      AS total_profit,
    ROUND(SUM(f.profit) / SUM(f.sales) * 100, 2) AS margin_pct,
    ROUND(AVG(f.profit_margin) * 100, 2)         AS avg_margin_pct,
    -- What percentage of transactions in this band are loss-making?
    ROUND(
        SUM(CASE WHEN NOT f.is_profitable THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*),
        1
    )                                            AS loss_making_pct,
    ROUND(SUM(f.discount_amount), 2)             AS total_discount_given
FROM sales_gold.fact_sales f
GROUP BY f.discount_band
ORDER BY avg_discount_pct

### Q5 — Top 10 Customers by Revenue
*"Who are our most valuable customers? Are they also profitable?"*

High-revenue customers are not always high-profit customers.
A customer who always demands deep discounts might generate
significant revenue but erode margins. This query surfaces both.

In [0]:
%sql
SELECT
    c.customer_name,
    c.segment,
    COUNT(DISTINCT f.order_id)                   AS total_orders,
    ROUND(SUM(f.sales), 2)                       AS total_revenue,
    ROUND(SUM(f.profit), 2)                       AS total_profit,
    ROUND(SUM(f.profit) / SUM(f.sales) * 100, 2) AS margin_pct,
    ROUND(AVG(f.discount) * 100, 1)              AS avg_discount_pct,
    ROUND(AVG(f.shipping_days), 1)               AS avg_shipping_days,
    MIN(f.order_date)                            AS first_order,
    MAX(f.order_date)                            AS last_order,
    -- Customer value tier
    CASE
        WHEN SUM(f.sales) >= 10000 THEN 'Platinum'
        WHEN SUM(f.sales) >= 5000  THEN 'Gold'
        WHEN SUM(f.sales) >= 1000  THEN 'Silver'
        ELSE                            'Bronze'
    END                                          AS customer_tier
FROM sales_gold.fact_sales    f
JOIN sales_gold.dim_customer  c ON f.customer_id = c.customer_id
GROUP BY
    c.customer_name,
    c.segment
ORDER BY total_revenue DESC
LIMIT 10

### Q6 — Products Losing the Most Money
*"Which products should be repriced, discontinued, or reviewed?"*

This query is operationally valuable — it gives the merchandising
team a ranked list of products to investigate. Loss-making products
are often the result of over-discounting, poor cost negotiation,
or misaligned pricing strategy.

In [0]:
%sql
SELECT
    p.product_name,
    p.category,
    p.sub_category,
    COUNT(*)                        AS times_sold,
    ROUND(SUM(f.sales), 2)          AS total_sales,
    ROUND(SUM(f.profit), 2)         AS total_profit,
    ROUND(AVG(f.discount) * 100, 1) AS avg_discount_pct,
    ROUND(
        SUM(f.profit) / SUM(f.sales) * 100,
        2
    )                               AS margin_pct
FROM sales_gold.fact_sales   f
JOIN sales_gold.dim_product  p ON f.product_id = p.product_id
GROUP BY
    p.product_name,
    p.category,
    p.sub_category
HAVING SUM(f.profit) < 0
ORDER BY total_profit ASC
LIMIT 15

### Q7 — Shipping Performance by Segment and Region
*"Are we delivering on our shipping commitments? Who waits longest?"*

Shipping performance is a customer satisfaction metric.
If Corporate customers wait longer than Consumer customers
despite paying for faster shipping, that is a service failure
worth surfacing to operations.

In [0]:
%sql
SELECT
    c.segment,
    f.region,
    f.ship_mode,
    COUNT(*)                            AS shipments,
    ROUND(AVG(f.shipping_days), 1)      AS avg_shipping_days,
    MIN(f.shipping_days)                AS fastest_days,
    MAX(f.shipping_days)                AS slowest_days,
    -- Percentage shipped within 3 days
    ROUND(
        SUM(CASE WHEN f.shipping_days <= 3 THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*),
        1
    )                                   AS within_3days_pct,
    f.shipping_speed,
    COUNT(DISTINCT f.order_id)          AS unique_orders
FROM sales_gold.fact_sales    f
JOIN sales_gold.dim_customer  c ON f.customer_id = c.customer_id
GROUP BY
    c.segment,
    f.region,
    f.ship_mode,
    f.shipping_speed
ORDER BY
    c.segment,
    avg_shipping_days DESC

### Q8 — Year-over-Year Quarterly Growth
*"Is each quarter growing compared to the same quarter last year?"*

YoY comparison removes seasonality bias. Q4 is always higher than Q3
in retail — comparing Q4 this year to Q3 this year tells you nothing.
Comparing Q4 this year to Q4 last year tells you whether you're growing.
This uses a self-join on the fact table — a common advanced SQL pattern.

In [0]:
%sql
WITH quarterly_sales AS (
    SELECT
        order_year,
        order_quarter,
        ROUND(SUM(sales), 2)  AS quarterly_sales,
        ROUND(SUM(profit), 2) AS quarterly_profit,
        COUNT(DISTINCT order_id) AS orders
    FROM sales_gold.fact_sales
    GROUP BY order_year, order_quarter
)
SELECT
    curr.order_year,
    curr.order_quarter,
    curr.quarterly_sales                        AS current_sales,
    prev.quarterly_sales                        AS prior_year_sales,
    ROUND(curr.quarterly_sales
          - prev.quarterly_sales, 2)            AS sales_delta,
    ROUND(
        (curr.quarterly_sales - prev.quarterly_sales)
        / prev.quarterly_sales * 100,
        2
    )                                           AS yoy_growth_pct,
    curr.orders                                 AS current_orders,
    CASE
        WHEN curr.quarterly_sales > prev.quarterly_sales THEN '📈 Growth'
        WHEN curr.quarterly_sales < prev.quarterly_sales THEN '📉 Decline'
        ELSE                                                   '➡️  Flat'
    END                                         AS trend
FROM quarterly_sales curr
LEFT JOIN quarterly_sales prev
    ON  curr.order_year    = prev.order_year + 1
    AND curr.order_quarter = prev.order_quarter
ORDER BY
    curr.order_year,
    curr.order_quarter

### Q9 — Customer Segment Deep Dive
*"Which segment is most valuable? Which is most price-sensitive?"*

Segmentation analysis drives marketing budget allocation.
If Corporate customers have 3x higher order values than Consumer
customers, that shapes where sales teams invest their time.

In [0]:
%sql
SELECT
    c.segment,
    COUNT(DISTINCT f.customer_id)                AS unique_customers,
    COUNT(DISTINCT f.order_id)                   AS total_orders,
    ROUND(SUM(f.sales), 2)                       AS total_sales,
    ROUND(SUM(f.profit), 2)                      AS total_profit,
    ROUND(SUM(f.profit) / SUM(f.sales) * 100, 2) AS margin_pct,
    ROUND(SUM(f.sales)
          / COUNT(DISTINCT f.customer_id), 2)    AS revenue_per_customer,
    ROUND(SUM(f.sales)
          / COUNT(DISTINCT f.order_id), 2)       AS avg_order_value,
    ROUND(AVG(f.discount) * 100, 1)              AS avg_discount_pct,
    ROUND(AVG(f.shipping_days), 1)               AS avg_shipping_days,
    -- Orders per customer (loyalty indicator)
    ROUND(
        COUNT(DISTINCT f.order_id) * 1.0
        / COUNT(DISTINCT f.customer_id),
        1
    )                                            AS orders_per_customer
FROM sales_gold.fact_sales   f
JOIN sales_gold.dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.segment
ORDER BY total_sales DESC

### Q10 — Executive Summary Dashboard
*"Give me the state of the business in one query."*

This is the query behind a CEO dashboard.
Every metric tells a story. Every number should prompt a question.
This is what data engineering ultimately delivers —
the foundation that makes business intelligence possible.

In [0]:
%sql
WITH base AS (
    SELECT
        f.*,
        c.segment,
        p.category
    FROM sales_gold.fact_sales   f
    JOIN sales_gold.dim_customer c ON f.customer_id = c.customer_id
    JOIN sales_gold.dim_product  p ON f.product_id  = p.product_id
)
SELECT
    -- Scale
    COUNT(DISTINCT order_id)                     AS total_orders,
    COUNT(DISTINCT customer_id)                  AS total_customers,
    COUNT(DISTINCT product_id)                   AS total_products,
    COUNT(*)                                     AS total_line_items,

    -- Revenue
    ROUND(SUM(sales), 2)                         AS total_revenue,
    ROUND(SUM(profit), 2)                        AS total_profit,
    ROUND(SUM(profit) / SUM(sales) * 100, 2)     AS overall_margin_pct,

    -- Efficiency
    ROUND(AVG(shipping_days), 1)                 AS avg_shipping_days,
    ROUND(AVG(discount) * 100, 1)                AS avg_discount_pct,
    ROUND(SUM(discount_amount), 2)               AS total_discount_given,

    -- Health
    ROUND(
        SUM(CASE WHEN is_profitable THEN 1 ELSE 0 END)
        * 100.0 / COUNT(*),
        1
    )                                            AS profitable_txn_pct,
    ROUND(SUM(sales) / COUNT(DISTINCT order_id), 2)
                                                 AS avg_order_value,
    ROUND(SUM(sales) / COUNT(DISTINCT customer_id), 2)
                                                 AS revenue_per_customer,

    -- Date range
    MIN(order_date)                              AS data_from,
    MAX(order_date)                              AS data_to
FROM base


## 4. Print the complete pipeline lineage

In [0]:
# Cell 13: Print the complete pipeline lineage
#
# This is the documentation a new engineer needs to understand
# the entire pipeline without reading every notebook.
# In production this would be generated automatically by
# a data lineage tool like OpenLineage or Databricks Unity Catalog.

print("=" * 65)
print("  SALES ETL PIPELINE — COMPLETE LINEAGE")
print("=" * 65)

layers = [
    ("SOURCE",    "Kaggle Superstore CSV",              "9,994 rows · 21 cols · raw"),
    ("BRONZE",    "sales_bronze.raw_superstore",        "snake_case cols · metadata added"),
    ("SILVER",    "sales_silver.clean_superstore",      "clean · typed · 13 derived cols"),
    ("SILVER DQ", "sales_silver.validation_results",    "53 checks · 100% pass rate"),
    ("GOLD FACT", "sales_gold.fact_sales",              "partitioned by year/month"),
    ("GOLD DIM",  "sales_gold.dim_customer",            "793 unique customers"),
    ("GOLD DIM",  "sales_gold.dim_product",             "1,862 products (SCD resolved)"),
    ("GOLD DIM",  "sales_gold.dim_region",              "geographic hierarchy"),
    ("ANALYTICS", "07_analytics_queries",               "10 business queries"),
]

for layer, name, note in layers:
    print(f"  {layer:<10} │ {name:<42} │ {note}")

print("=" * 65)
print()
print("  Architecture  : Medallion (Bronze → Silver → Gold)")
print("  Format        : Delta Lake (ACID, time travel)")
print("  Compute       : Databricks Serverless")
print("  Partitioning  : fact_sales by order_year + order_month")
print("  Quality gate  : 53 automated checks in Notebook 04")
print("  Star schema   : 1 fact + 3 dimensions")
print("=" * 65)

log("INFO", "07_analytics", "Pipeline complete ✅")


## 5. Project completion

In [0]:
# Cell 14: Project completion

print("""
╔═══════════════════════════════════════════════════════════╗
║         SALES ETL PIPELINE — PROJECT COMPLETE             ║
╠═══════════════════════════════════════════════════════════╣
║                                                           ║
║  ✅ Notebook 00  Setup and configuration                  ║
║  ✅ Notebook 01  Bronze ingestion (CSV → Delta)           ║
║  ✅ Notebook 02  Silver cleaning (nulls, types, dedup)    ║
║  ✅ Notebook 03  Silver transformation (13 derived cols)  ║
║  ✅ Notebook 04  Silver validation (53 DQ checks)         ║
║  ✅ Notebook 05  Gold fact table (fact_sales)             ║
║  ✅ Notebook 06  Gold dimensions (customer/product/region)║
║  ✅ Notebook 07  Analytics (10 business queries)          ║
║                                                           ║
║  Tech stack : PySpark · Spark SQL · Delta Lake            ║
║  Pattern    : Medallion Architecture · Star Schema        ║
║  Platform   : Databricks Serverless                       ║
║                                                           ║
╚═══════════════════════════════════════════════════════════╝
""")